# Qualitative results: ligand & pocket

Rows represent pockets; columns show **Reference → VoxBind → VoxBind + Ours → additional baselines**. The molecular meshes, colors, lighting, and default camera are taken directly from `fig1.ipynb`. No density, occupancy surfaces, or voxel lattice are displayed.

1. Choose methods and pockets below, then run the cells in order.
2. Use each row's pocket selector and each method's ligand selector. Indices are **zero-based record indices in the original SDF**; individual SDF filenames are also shown.
3. Rotate or zoom any panel to **synchronize the camera across its row**. Click **Inspect selected ligands** to see 2D structures, SMILES, and source paths.
4. Run the last cell to save the current selection and cameras as PNG, PDF, SVG, and a reproducibility JSON.

Default pockets are the first two IDs with SDF files for every selected method. Each default ligand is the first readable record; examples are not ranked by performance. Inspect and select the examples you want to use in the final figure.


## Kernel / dependencies

On this server, use the `voxbind` Python 3.12 kernel with the prepared `.cache/fig-qual` runtime. In another environment, use a Jupyter kernel with RDKit installed. If necessary, run the following commands **in that kernel**, then restart it.

```python
%pip install "plotly>=6.1.1" ipywidgets anywidget "kaleido>=1" nbformat pillow
# If RDKit is missing: %pip install rdkit
# Only if Kaleido cannot find Chrome:
# import plotly.io as pio
# pio.get_chrome()
```

Plotly 3D scenes use WebGL. Molecular scenes are embedded as raster images in PDF/SVG, while titles and labels remain vector text. Set `EXPORT_SCALE` in the last cell to control resolution.


In [1]:
from pathlib import Path
import json
import sys
import os

# Works from the repository root, notebook directory, or figures directory.
REPO_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "notebook/figures/fig1.ipynb").is_file()), None
)
if REPO_ROOT is None:
    raise FileNotFoundError("Start the kernel inside the VoxBind repository.")
# Use this server's isolated runtime cache without changing conda packages.
# If absent or using another Python version, follow the dependency instructions above.
LOCAL_RUNTIME = REPO_ROOT / ".cache/fig-qual"
if sys.version_info[:2] == (3, 12) and (LOCAL_RUNTIME / "python").is_dir():
    local_python = str(LOCAL_RUNTIME / "python")
    if local_python not in sys.path:
        sys.path.insert(0, local_python)
    # Kaleido launches a Python wrapper that also needs the cached dependencies.
    if local_python not in os.environ.get("PYTHONPATH", "").split(os.pathsep):
        os.environ["PYTHONPATH"] = os.pathsep.join(
            p for p in (local_python, os.environ.get("PYTHONPATH", "")) if p
        )
local_chrome = LOCAL_RUNTIME / "chrome/chrome-linux64/chrome"
if local_chrome.is_file():
    os.environ.setdefault("BROWSER_PATH", str(local_chrome))
    local_libs = str(LOCAL_RUNTIME / "browser-libs/usr/lib/x86_64-linux-gnu")
    if local_libs not in os.environ.get("LD_LIBRARY_PATH", "").split(os.pathsep):
        os.environ["LD_LIBRARY_PATH"] = os.pathsep.join(
            p for p in (local_libs, os.environ.get("LD_LIBRARY_PATH", "")) if p
        )

local_fonts = LOCAL_RUNTIME / "fonts.conf"
if local_fonts.is_file():
    os.environ.setdefault("FONTCONFIG_FILE", str(local_fonts))

import plotly.io as pio
from IPython.display import display

FIGURE_DIR = REPO_ROOT / "notebook/figures"
if str(FIGURE_DIR) not in sys.path:
    sys.path.insert(0, str(FIGURE_DIR))
from qual_utils import Catalog, QualitativeGrid, fig1_style, make_table, method_specs

BASELINE_ROOT = REPO_ROOT.parent / "base_drug"
STYLE = fig1_style(FIGURE_DIR / "fig1.ipynb")
METHOD_SPECS = method_specs(REPO_ROOT, BASELINE_ROOT)
catalog = Catalog(METHOD_SPECS)
display(make_table(catalog.coverage()))


HTML(value='<table style="border-collapse:collapse;text-align:left;font-size:12px"><tr><th style=\'padding:5px…

## Methods & pockets

`METHODS` sets the column order after Reference. Additional SDF baselines on this server are TargetDiff, FuncBind, AR, and Pocket2Mol. Optional runs include `"VoxBind + Ours v2"` and `"VoxBind σ=1.0"`. FuncBind GPU shards are combined into one method. DecompDiff currently has raw sampling output but no SDF files, so its coverage is zero.

To register another method, add its `root` and `layout` to `METHOD_SPECS`, then recreate the catalog. Supported layouts: `eval` (`target_XX/samples.sdf`), `eval_shards` (`gpuN/samples/target_XX/samples.sdf`), and `sweep` (`id_N/run/SDF/*.sdf`).

AR/Pocket2Mol use the server's verified common CrossDocked mapping, `id_N == target_N` (see `base_drug/export_targetdiff_sdf.py`). If multiple runs exist, the last run name containing SDF files is used; runs are never pooled. Exact source paths are available through Inspect and the saved JSON. Reference and pocket files come from Ours v1. Other eval-format methods are checked for matching reference identity, reference coordinates, and pocket coordinates.


In [ ]:
METHODS = ["VoxBind", "VoxBind + Ours", "TargetDiff", "FuncBind", "AR", "Pocket2Mol"]
# Example: METHODS = ["VoxBind", "VoxBind + Ours", "TargetDiff"]
# Example: METHODS += ["VoxBind + Ours v2"]

COMMON_TARGETS = catalog.common_targets(METHODS)
print(f"Common pockets ({len(COMMON_TARGETS)}): {', '.join(COMMON_TARGETS)}")
if len(COMMON_TARGETS) < 2:
    raise ValueError("At least two pockets with SDF files for every selected method are required. Check METHODS and paths.")

POCKET_IDS = COMMON_TARGETS[:2]  # Current server defaults: target_03, target_05
# POCKET_IDS = ["target_03", "target_05"]  # Pin IDs here or change them in the UI.
PANEL_PX = 310
POCKET_HALF_EXTENT_A = 8.0  # Pocket heavy atoms in fig1's reference-centered 16 Å cube.

# Optional: restore methods, pockets, records, cameras, and panel size from a previous export.
RESTORE_JSON = None  # Example: FIGURE_DIR / "exports/fig-qual/fig-qual.selection.json"
RESTORED = json.loads(Path(RESTORE_JSON).read_text()) if RESTORE_JSON else {}
if RESTORED:
    METHODS = RESTORED["methods"]
    POCKET_IDS = RESTORED["targets"]
    PANEL_PX = RESTORED["panel_px"]
    POCKET_HALF_EXTENT_A = RESTORED["pocket_half_extent_A"]
print("Rows:", POCKET_IDS)
print("Columns:", ["Reference", *METHODS])


## Interactive comparison

Ligand: `#F5B27E` · Pocket: `#8291E8` · ball/stick radius: `0.32 Å` · white background.

The reference ligand's heavy-atom centroid is subtracted **identically from every ligand and the pocket**. Generated ligands are never individually aligned or recentered. Original SDF poses are shown without redocking. The pocket crop stays fixed within each row; ligands are never cropped. Selecting a large ligand expands the axis range equally across the row.

Only readable 3D records appear in the ligand menus; skipped counts are shown. Scroll horizontally if necessary. Changing a ligand rebuilds that row and retains its camera. Inspect computes 2D depictions on copies, preserving the original 3D coordinates.


In [ ]:
# Rerunning this cell rebuilds the UI from the settings above.
grid = QualitativeGrid(
    catalog, STYLE, METHODS, POCKET_IDS,
    panel_px=PANEL_PX, pocket_extent=POCKET_HALF_EXTENT_A,
    selections=RESTORED.get("selections"), cameras=RESTORED.get("cameras"),
)
display(grid.widget)


## Check the current selection

Inspect the original SDF paths and record indices selected in the UI. `centroid_offset_A` is the distance from the reference centroid; offsets above 10 Å are flagged for inspection. This distance is neither an alignment metric nor a docking score. Rerun this cell after changing the selection to refresh the table.


In [ ]:
display(make_table(grid.selection_report()))
# Inspect the current reproducibility settings in memory.
selection = grid.manifest()


## Export — run this cell last

The current **pockets, per-method ligands, rotation, and zoom** are read directly from the live UI. You do not need to rerun the selection-report cell. Existing exports with the same name are overwritten; change `OUTPUT_STEM` to keep another version.


In [ ]:
OUTPUT_DIR = FIGURE_DIR / "exports/fig-qual"
OUTPUT_STEM = "fig-qual"
EXPORT_SCALE = 3  # Resolution multiplier for PNG and embedded WebGL scenes.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

final_fig = grid.figure()  # Snapshot the current live selection and cameras.
export_paths = [OUTPUT_DIR / f"{OUTPUT_STEM}.{ext}" for ext in ("png", "pdf", "svg")]
try:
    pio.write_images(
        fig=[final_fig] * len(export_paths),
        file=[str(p) for p in export_paths],
        format=["png", "pdf", "svg"], scale=EXPORT_SCALE,
        width=final_fig.layout.width, height=final_fig.layout.height,
    )
except Exception as exc:
    raise RuntimeError(
        "Image export failed. This kernel needs recent plotly/kaleido, Chrome, "
        "and Chrome's shared libraries. See the dependency instructions above. Cause: " + str(exc)
    ) from exc

manifest_path = OUTPUT_DIR / f"{OUTPUT_STEM}.selection.json"
manifest_path.write_text(json.dumps(grid.manifest(), indent=2, ensure_ascii=False) + "\n")
for path in [*export_paths, manifest_path]:
    print(f"Saved: {path} ({path.stat().st_size:,} bytes)")
